# ebook2audiobook - Android APK build (Google Colab)

Buildozer only runs on Linux, so the APK for the thin Android client
(`android_client/`) is built here on a free Colab machine.

## How to use
1. Runtime type: plain **CPU** is enough (no GPU needed).
2. Run the two cells top to bottom. The first build takes ~30-50 min
   (Android SDK/NDK download); repeat builds are much faster.
3. The finished `ebook2audiobook-*.apk` downloads automatically.
4. On the phone (Android 10/11): allow installing from unknown sources,
   open the APK, done.

## Updates
The APK is signed with a **permanent keystore** that lives on your Google
Drive (shared as a link; cell 2 downloads it automatically via `gdown` -
it is **not** stored in git for security). Because every build shares the
same signature, installing a newer APK over an older one is a normal
**update** - no need to uninstall and reinstall. Keep that keystore
**secret** and never lose it: if it is replaced, phones will refuse an
in-place update. Bump `version` / `version.code` in
`android_client/buildozer.spec` for each release so Android sees a higher
version.

If the build fails, cell (2) prints the error lines from `build.log`
automatically - copy them when reporting a problem.

In [ ]:
#@title (1) Get the client sources & install buildozer
REPO = 'Tarkas/Book-to-audiobook'  #@param {type:"string"}
BRANCH = 'master'  #@param {type:"string"}

import os, subprocess, sys

cmds = [
    'sudo apt-get update -qq',
    # autotools/gettext family is needed to build libffi & friends
    'sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf automake autopoint '
    'libtool libtool-bin libltdl-dev gettext patch pkg-config ccache '
    'zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev',
    # cython 0.29.37 is the newest 0.29.x and supports Python 3.12 (current Colab)
    'pip install -q buildozer cython==0.29.37',
]
for c in cmds:
    print('\n$', c)
    r = subprocess.run(c, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'Setup command failed (exit {r.returncode}): {c}')

if not os.path.isdir('ebook2audiobook'):
    r = subprocess.run(
        f'git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git ebook2audiobook',
        shell=True)
    if r.returncode != 0:
        raise SystemExit('git clone failed - check REPO/BRANCH above')
else:
    # Reused runtime: fetch the requested branch and hard-switch to it, so a
    # BRANCH change (e.g. main -> master) actually takes effect. `git pull`
    # alone would stay on the previously checked-out branch (usually main)
    # and would silently build the WRONG code. The compiled p4a recipes in
    # .buildozer are cached separately and are NOT affected by the branch
    # switch, so a re-run still picks up fixes without a full rebuild.
    r = subprocess.run(
        'git -C ebook2audiobook fetch --depth 1 origin ' + BRANCH, shell=True)
    if r.returncode != 0:
        print('git fetch failed - consider deleting the runtime and rerunning.')
    else:
        r = subprocess.run(
            'git -C ebook2audiobook checkout -B ' + BRANCH + ' origin/' + BRANCH,
            shell=True)
        if r.returncode != 0:
            print('git checkout failed - consider deleting the runtime and rerunning.')

%cd ebook2audiobook/android_client
print('\nSetup done. Python:', sys.version.split()[0])

In [ ]:
#@title (2) Build the APK & download
import glob, os, re, subprocess

# Gradle / Android toolchain needs Java 17 (Colab default may be 11)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# The permanent signing key is NOT stored in git for security. It lives
# only on Google Drive and is downloaded here via gdown (preinstalled in
# Colab), so every build shares the SAME signature - that is what lets
# phones install a new APK as an UPDATE instead of uninstall/reinstall.
# IMPORTANT: the Drive file must be shared as 'Anyone with the link'.
KEY_FILE_ID = '1k_I2BLE7irwxm26OauoNoHgShD4sYlDo'
if not os.path.exists('ebook2audiobook-release.keystore'):
    print('No local keystore - downloading it from Google Drive...')
    rc = subprocess.run(
        'gdown --id ' + KEY_FILE_ID + ' -O ebook2audiobook-release.keystore',
        shell=True).returncode
    if rc != 0 or not os.path.exists('ebook2audiobook-release.keystore'):
        raise SystemExit(
            'Failed to download keystore from Google Drive (file id ' +
            KEY_FILE_ID + ').\n'
            'Make sure the file is shared as \'Anyone with the link\'.\n'
            'Without it a fresh build uses a per-machine debug key and '
            'phones would refuse to update in place.')
    os.chmod('ebook2audiobook-release.keystore', 0o600)
else:
    print('Using keystore from repo/local copy.')

# buildozer downloads the Android SDK/NDK on first run (long, one-off).
# Full output goes to build.log; only progress markers are shown here.
# NOTE: "release" (not "debug") so buildozer signs with the committed
# ebook2audiobook-release keystore configured in buildozer.spec.
progress = re.compile(
    r'(-> running|# Prepar|# Build|# Download|# Install|# Unpack|# Compil|'
    r'BUILD (SUCCESSFUL|FAILED)|STDERR|[Ee]rror|FAILED|Exception|'
    r'sign|Signing)')
with open('build.log', 'w', errors='replace') as log:
    proc = subprocess.Popen('yes | buildozer -v android release', shell=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors='replace')
    for line in proc.stdout:
        log.write(line)
        if progress.search(line):
            print(line, end='')
    proc.wait()
print('\nbuildozer exit code:', proc.returncode)

apks = sorted(glob.glob('bin/*.apk'))
if apks:
    apk = apks[-1]
    print('APK ready:', apk)
    try:
        from google.colab import files
        files.download(apk)
    except Exception as e:
        print('Auto-download failed:', e, '- grab the file from the Files panel:', apk)
else:
    lines = open('build.log', errors='replace').read().splitlines()
    print('\n--- BUILD FAILED, last 80 lines of build.log: ---')
    print('\n'.join(lines[-80:]))
    # Selecting/copying cell output is painful (especially on phones), so
    # pack a report and download it instead of asking anyone to copy text.
    with open('build_report.txt', 'w', errors='replace') as rep:
        rep.write('buildozer exit code: %s\n\n' % proc.returncode)
        fails = [i for i, l in enumerate(lines)
                 if re.search(r'# Command failed|BUILD FAILED|Aborted!', l)]
        # The same marker can repeat thousands of times (once per packaged
        # file) and blow the report up to tens of MB - the tail is enough.
        for i in fails[-10:]:
            rep.write('=' * 70 + '\n')
            rep.write('\n'.join(lines[max(0, i - 60):i + 15]) + '\n')
        rep.write('=' * 70 + '\n--- last 200 lines ---\n')
        rep.write('\n'.join(lines[-200:]) + '\n')
    subprocess.run('zip -q -9 build_log.zip build.log', shell=True)
    try:
        from google.colab import files
        files.download('build_report.txt')
        files.download('build_log.zip')
        print('\nDownloaded build_report.txt + build_log.zip - send these files.')
    except Exception as e:
        print('Auto-download failed:', e,
              '- grab build_report.txt / build_log.zip from the Files panel.')
    err = [l for l in lines if re.search(r'error:|Error:|FAILED|Exception|No such file|not found', l)]
    if err:
        print('\n--- lines mentioning errors (copy these when reporting): ---')
        print('\n'.join(err[-40:]))

In [ ]:
#@title (3) Diagnostics - real failures only + tail of build.log
import glob, re

apks = sorted(glob.glob('bin/*.apk'))
print('APK files:', apks if apks else 'NONE')

lines = open('build.log', errors='replace').read().splitlines()

# Real failure markers only. p4a's "Trying first build ... this is expected
# to fail" passes produce harmless clang/ccache errors - do not report them.
real = [i for i, l in enumerate(lines)
        if re.search(r'# Command failed|BUILD FAILED|Aborted!|buildozer.*[Ee]rror', l)]
if real:
    for i in real:
        lo, hi = max(0, i - 40), min(len(lines), i + 10)
        print('=' * 70)
        print('\n'.join(lines[lo:hi]))
    print('=' * 70)
    print('Copy everything above when reporting the problem.')
else:
    print('No real failure markers found.')

print('\n--- last 60 lines of build.log ---')
print('\n'.join(lines[-60:]))

# Save the same diagnostics to a file and download it: no copying needed.
with open('build_report.txt', 'w', errors='replace') as rep:
    for i in real[-10:]:
        rep.write('=' * 70 + '\n')
        rep.write('\n'.join(lines[max(0, i - 40):i + 10]) + '\n')
    rep.write('=' * 70 + '\n--- last 200 lines ---\n')
    rep.write('\n'.join(lines[-200:]) + '\n')
try:
    from google.colab import files
    files.download('build_report.txt')
    print('Downloaded build_report.txt - send this file.')
except Exception as e:
    print('Auto-download failed:', e, '- grab build_report.txt from the Files panel.')